# OpenFEMA API Tutorial: Part 7 - Joining With Non-FEMA Data

## Quick Summary
- This tutorial demonstrates how to combine data retrieved from the OpenFEMA API with data from other Federal Government data sources.
- Integral to joining with external data is a key or shared field that exists in both data sources. Often the values in that shared field must be normalized or transformed to facilitate the join.
- From a data science perspective, more value can be exracted from OpenFEMA data when combined with outside information. This notebook works through seven external sources:
  * **U.S. Census Bureau** - responsible for producing data about the American people and economy
  * **SAM.gov** (System for Award Management) - entity registration, contract opportunities, and federal award data
  * **USAspending.gov** - publishes federal spending data, including contracts, grants, loans, and direct payments
  * **Federal Register** - federal government’s official daily journal, containing agency rules, notices, and presidential documents
  * **Weather.gov** (National Weather Service) - provides weather, water, climate data, forecasts, and warnings 
  * **NOAA NCEI** (National Centers for Environmental Information) - manages a large archive of atmospheric, oceanic, coastal, and geophysical environmental data
  * **USGS** (United States Geological Survey) - data related natural hazards, resources, ecosystems, and Earth systems 

## Example Table of Contents
  * [ 1) Census - Assistance Dollars Per Capita](#1%29-Census---Assistance-Dollars-Per-Capita)
  * [ 2) Census - Joining on FIPS Code Instead of County Name](#2%29-Census---Joining-on-FIPS-Code-Instead-of-County-Name)
  * [ 3) SAM.gov - Enriching Public Assistance Applicants](#3%29-SAM.gov---Enriching-Public-Assistance-Applicants)
  * [ 4) USAspending.gov - Identifying FEMA Programs and Award Types](#4%29-USAspending.gov---Identifying-FEMA-Programs-and-Award-Types)
  * [ 5) USAspending.gov - PA Disaster Grants](#5%29-USAspending.gov---PA-Disaster-Grants)
  * [ 6) Federal Register - Disaster Notices and Amendments](#6%29-Federal-Register---Disaster-Notices-and-Amendments)
  * [ 7) Federal Register - Declarations in a Date Range, Enriched with FEMA Data](#7%29-Federal-Register---Declarations-in-a-Date-Range,-Enriched-with-FEMA-Data)
  * [ 8) Weather.gov - IPAWS Alerts Issued for a UGC](#8%29-Weather.gov---IPAWS-Alerts-Issued-for-a-UGC)
  * [ 9) Weather.gov - Retrieving an NWS Alert from an IPAWS Alert](#9%29-Weather.gov---Retrieving-an-NWS-Alert-from-an-IPAWS-Alert)
  * [10) NCEI - Historical Weather Events by Disaster Declaration](#10%29-NCEI---Historical-Weather-Events-by-Disaster-Declaration)
  * [11) USGS - Earthquake Events by Disaster Declaration](#11%29-USGS---Earthquake-Events-by-Disaster-Declaration)
  * [12) USGS - Water Level by Disaster and County](#12%29-USGS---Water-Level-by-Disaster-and-County)
  

## Overview
Previous tutorials have demonstrated the basic use of the OpenFEMA API - some joining multiple OpenFEMA datasets. The focus of this notebook is enriching data with other agency API's. Some of these API's require subscription keys while others are completely open.

<div class="alert alert-block alert-info">
<strong>Note:</strong> This tutorial uses two external libraries - <code>requests</code> for calling the APIs and <code>pandas</code> for holding and joining the data. Other Python libraries are available that offer the same capability.
</div>

## A Note on Join Keys
The single most important and most error-prone part of combining data from different sources is identifying and matching a data element or join keys. A join key may have the same name across data sources but may not actually match due to differences in punctuation, capitalization, abbreviations, and inclusion of other data. Where possible, try to utilize standardized codes for joining. 

## 1) Census - Assistance Dollars Per Capita
Census data is often used to enrich data with geographic information and demographic statistics. This example performs a **bulk join**. We will download an entire external table (county populations from the U.S. Census Bureau API) and match it, row for row, to a summary of Individual Assistance data. We will utilize Census data to express IA dollars per unit of population.

### Retrieve the OpenFEMA Data
First we retrieve the FEMA data we want to enrich. We ask the IndividualsAndHouseholdsProgramValidRegistrations endpoint for Hurricane Laura (disaster 4559) records in Louisiana, selecting only the parish name and damage amount fields we need.

<div class="alert alert-block alert-info">
<strong>Note:</strong> This example uses the <code>$allrecords</code> parameter as recommended by OpenFEMA rather than paging to retrieve the data.
</div>

In [7]:
# imports
from datetime import datetime, timedelta, timezone
import math
import re
import requests
import pandas as pd

# setting global formatting for dataframes so dollar amounts don't show as scientific notation
pd.options.display.float_format = '${:,.2f}'.format

In [8]:
# define the OpenFEMA API query - IHP Valid Registrations for Hurricane Laura (disaster 4559) in Louisiana
# This is a pretty large request so it may take a second, or you may have to retry
baseURL = "https://www.fema.gov/api/open/v2/IndividualsAndHouseholdsProgramValidRegistrations?"
apiFilter = "$filter=damagedStateAbbreviation%20eq%20%27LA%27%20and%20disasterNumber%20eq%204559"
apiSelect = "$select=county,rpfvl,ppfvl"
apiParams = "&$metadata=off&$format=jsona&$allrecords=true"


try:
    response = requests.get(baseURL + apiFilter + '&' + apiSelect + apiParams, timeout=300)
    jsonDataAll = response.json()
except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError) as err:
    print("download interrupted (" + type(err).__name__ + ")")


# load the retrieved records into a pandas dataframe
df = pd.DataFrame(jsonDataAll)
print(str(len(df)) + ' records retrieved')
df.head()

216709 records retrieved


,county,rpfvl,ppfvl
0,Calcasieu (Parish),$0.00,$0.00
1,Calcasieu (Parish),$0.00,$0.00
2,Calcasieu (Parish),$0.00,$0.00
3,Sabine (Parish),$0.00,$0.00
4,Calcasieu (Parish),$0.00,"$1,871.90"


### Summarize the OpenFEMA Data
This dataset returns data at a "registrant" level - each row accounting for one household's valid registration. As we are interested in total assistance dollars per parish, we must group the records by the ```county``` field and sum the real-property (```rpfvl```) and personal-property (```ppfvl```) values. This produces one row per parish, the granularity we need to join to one population figure per parish.

In [9]:
dfSumCounty = df.groupby(
    ['county']
).agg(
    # real property components
    totRpfvl=('rpfvl', 'sum'),
    # personal property components
    totPpfvl=('ppfvl', 'sum')
)
dfSumCounty['totCost'] = round(dfSumCounty.totRpfvl + dfSumCounty.totPpfvl)

dfSumCounty

,totRpfvl,totPpfvl,totCost
county,,,
Acadia (Parish),"$2,683,652.37","$976,810.22","$3,660,463.00"
Allen (Parish),"$7,033,009.46","$1,275,260.58","$8,308,270.00"
Beauregard (Parish),"$13,382,632.99","$2,617,804.29","$16,000,437.00"
Caddo (Parish),"$2,500,055.84","$3,077,236.69","$5,577,293.00"
Calcasieu (Parish),"$101,267,938.79","$55,429,421.86","$156,697,361.00"
Cameron (Parish),"$19,997,207.82","$4,637,071.43","$24,634,279.00"
Grant (Parish),"$3,842,858.11","$656,993.04","$4,499,851.00"
Jackson (Parish),"$803,068.02","$146,173.54","$949,242.00"
Jefferson Davis (Parish),"$6,675,468.85","$1,931,066.76","$8,606,536.00"


### Retrieve the Non-FEMA Data
To put the FEMA assistance dollars in context, we pull county population counts from the [**U.S. Census Bureau API**](https://www.census.gov/data/developers/data-sets.html), our non-FEMA source. We use the 2020 Census (```P1_001N```, total population) and request every county in Louisiana (state 22).

<div class="alert alert-block alert-warning">
<strong>Requires an API key:</strong> The Census Bureau API requires a free API key. Without one, requests are rejected with a "Missing Key" response. Request a key at <a href="https://api.census.gov/data/key_signup.html">https://api.census.gov/data/key_signup.html</a> and paste it into the <code>CENSUS_API_KEY</code> variable below.
</div>

A join only works when the shared key holds identical values in both datasets. The Census returns names like ```Acadia Parish, Louisiana``` while FEMA uses ```Acadia (Parish)```, so we reformat the Census names to match the FEMA convention before joining.

In [13]:
# paste your free Census Bureau API key here (request one at https://api.census.gov/data/key_signup.html)
CENSUS_API_KEY = "YOUR_KEY_HERE"

# get 2020 total population (P1_001N) for every county in Louisiana (state 22) from the Census Bureau API
censusURL = "https://api.census.gov/data/2020/dec/pl"
censusParams = {
    "get": "NAME,P1_001N",
    "for": "county:*",
    "in": "state:22",
    "key": CENSUS_API_KEY
}
response = requests.get(censusURL, params=censusParams)
jsonData = response.json()

# the Census API returns a list of lists, the first row is the header, the rest are data.
# it also returns 'state' and 'county' FIPS-code columns, keep only NAME + population BEFORE renaming,
# otherwise renaming NAME to 'county' collides with the FIPS 'county' column and the join fails.
dfPop = pd.DataFrame(jsonData[1:], columns=jsonData[0])
dfPop = dfPop[["NAME", "P1_001N"]].rename(columns={"NAME": "county", "P1_001N": "population"})

# population is a string, convert it to a number so we can do arithmetic later
dfPop["population"] = pd.to_numeric(dfPop["population"])

# reformat the Census county names to match the FEMA 'county' values so the join keys line up
dfPop["county"] = dfPop["county"].str.replace(", Louisiana", "")
dfPop["county"] = dfPop["county"].str.replace("Parish", "(Parish)")
dfPop["county"] = dfPop["county"].str.replace("LaSalle", "La Salle")

dfPop.head()

,county,population
0,Acadia (Parish),57576
1,Allen (Parish),22750
2,Ascension (Parish),126500
3,Assumption (Parish),21039
4,Avoyelles (Parish),39693


### Join the Two Datasets
With both datasets sharing a common, identically-formatted ```county``` key, we can combine them. Here we use the pandas ```join``` method to attach each parish's population to its FEMA dollar summary. Once the two sources are combined, we compute a value neither dataset held, assistance dollars **per capita**.

In [14]:
# join on the county name to attach each parish's population to its FEMA dollar summary
dfJoined = dfSumCounty.join(dfPop.set_index('county'), on='county')

# Compute per capita cost
dfJoined['costPerCapita'] = round(dfJoined['totCost'] / dfJoined['population'], 2)

dfJoined[['totCost', 'population', 'costPerCapita']].sort_values('costPerCapita', ascending=False)

,totCost,population,costPerCapita
county,,,
Cameron (Parish),"$24,634,279.00",5617,"$4,385.66"
Calcasieu (Parish),"$156,697,361.00",216785,$722.82
Beauregard (Parish),"$16,000,437.00",36549,$437.78
Allen (Parish),"$8,308,270.00",22750,$365.20
Jefferson Davis (Parish),"$8,606,536.00",32250,$266.87
Grant (Parish),"$4,499,851.00",22169,$202.98
Vernon (Parish),"$6,937,176.00",48750,$142.30
Winn (Parish),"$1,724,834.00",13755,$125.40
Rapides (Parish),"$16,296,492.00",130023,$125.34


## 2) Census - Joining on FIPS Code Instead of County Name
Census Example 1 worked, but only after a bit of name transformation. This example builds a similar per-capita measure with **no name matching at all**, by joining on **FIPS codes**, the standardized federal identifiers for states and counties (Louisiana is ```22```, Calcasieu Parish is ```019```). Both sides of the join also switch to new datasets:

- **FEMA side**: ```PublicAssistanceFundedProjectsDetails``` - one row per funded Public Assistance project, carrying a FIPS ```countyCode``` right next to each ```federalShareObligated``` dollar amount
- **Census side**: the **American Community Survey (ACS) 5-year estimates**. The ACS is a continuous survey that estimates the economic status the head count doesn't collect, such as **median household income**.

Joining the two produces a table showing federal PA dollars **per resident**, with each parish's income alongside for context

Codes still demand care, just less so. FEMA stores the county code unpadded (```'19'```) while the Census pads it (```'019'```). Projects run by state agencies are labeled ```Statewide``` but still carry a home-parish county code, so they must be filtered out as well.

Our results show that Cameron Parish had the most public assistance funding per capita, which lines up with Laura's path and Cameron's low population count.

In [15]:
# get every Public Assistance funded project for Hurricane Laura (4559)
paProjURL = "https://www.fema.gov/api/open/v2/PublicAssistanceFundedProjectsDetails?"
paProjFilter = "$filter=disasterNumber%20eq%204559"
paProjParams = "&$select=county,countyCode,stateNumberCode,federalShareObligated&$format=jsona&$metadata=off&$allrecords=true"
dfProj = pd.DataFrame(requests.get(paProjURL + paProjFilter + paProjParams).json())

# filter statewide projects
dfStatewide = dfProj[dfProj["county"] == "Statewide"]
dfProj = dfProj[dfProj["county"] != "Statewide"]
print(str(len(dfStatewide)) + " statewide projects excluded ($" +
      str(round(dfStatewide["federalShareObligated"].sum())) + " federal share)")

# sum the federal share by parish & fix code padding issue
dfProj["fipsCountyCode"] = dfProj["countyCode"].str.zfill(3)
dfProj["fipsStateCode"] = dfProj["stateNumberCode"].str.zfill(2)
dfPA = dfProj.groupby(["fipsStateCode", "fipsCountyCode"], as_index=False).agg(
    paFederalShare=("federalShareObligated", "sum"))

# get median household income and total population for every Louisiana county in one ACS call
acsURL = "https://api.census.gov/data/2021/acs/acs5"
acsParams = {
    "get": "NAME,B19013_001E,B01003_001E",  # name, median household income, total population
    "for": "county:*",
    "in": "state:22",
    "key": CENSUS_API_KEY
}
acsJson = requests.get(acsURL, params=acsParams).json()
dfACS = pd.DataFrame(acsJson[1:], columns=acsJson[0]).rename(columns={
    "NAME": "parish", "B19013_001E": "medianIncome",
    "B01003_001E": "population", "state": "fipsStateCode", "county": "fipsCountyCode"})
dfACS["medianIncome"] = pd.to_numeric(dfACS["medianIncome"])
dfACS["population"] = pd.to_numeric(dfACS["population"])
dfACS["parish"] = dfACS["parish"].str.replace(", Louisiana", "")  # cosmetic only

# merge on the FIPS codes
dfPACapita = dfPA.merge(dfACS, on=["fipsStateCode", "fipsCountyCode"])
dfPACapita["paPerCapita"] = round(dfPACapita["paFederalShare"] / dfPACapita["population"], 2)

print(str(len(dfPACapita)) + " of " + str(len(dfPA)) + " parishes matched")
dfPACapita[["parish", "fipsCountyCode", "paFederalShare", "medianIncome", "population",
            "paPerCapita"]].sort_values("paPerCapita", ascending=False).head(10)

340 statewide projects excluded ($1408520419 federal share)
54 of 54 parishes matched


,parish,fipsCountyCode,paFederalShare,medianIncome,population,paPerCapita
10,Cameron Parish,023,"$97,353,665.34",64525,5650,"$17,230.74"
8,Calcasieu Parish,019,"$1,406,546,732.10",59470,212646,"$6,614.50"
4,Beauregard Parish,011,"$114,500,730.73",57130,36417,"$3,144.16"
53,Winn Parish,127,"$11,973,907.30",43710,13875,$862.98
1,Allen Parish,003,"$18,354,398.14",47660,23085,$795.08
17,Grant Parish,043,"$16,685,802.31",52779,22236,$750.40
48,Vermilion Parish,113,"$25,931,787.46",56347,57775,$448.84
22,Jefferson Davis Parish,053,"$14,224,128.24",45578,32270,$440.78
49,Vernon Parish,115,"$15,709,019.59",53215,49064,$320.17
36,Sabine Parish,085,"$6,557,369.36",39975,22377,$293.04


## 3) SAM.gov - Enriching Public Assistance Applicants

Joining with external sources may not have to be done as a bulk join. Sometimes you just need to lookup a few values one record at a time to enrich the OpenFEMA data. 

In this example, we will enrich FEMA **Public Assistance (PA) Applicants** with registration details from the [**U.S. General Services Administration's SAM.gov Entity Management API**](https://open.gsa.gov/api/entity-api/). Any organization that receives federal funding is expected to maintain an active registration in the System for Award Management (SAM.gov). For each applicant we will look up its Unique Entity Identifier (UEI), CAGE code, and registration status.

### Retrieve FEMA PA Applicants
We start with the organizations that applied for Public Assistance under Hurricane Laura (disaster 4559) in Louisiana. To keep the number of external lookups small (see the rate-limit note in the next step), we retrieve just the first few applicants.

In [7]:
# retrieve a few Public Assistance applicants for Hurricane Laura (disaster 4559) in Louisiana.
paURL = "https://www.fema.gov/api/open/v1/PublicAssistanceApplicants?"
paFilter = "$filter=disasterNumber%20eq%204559"
paParams = "&$select=applicantName,city&$top=3&$format=jsona&$metadata=off"

response = requests.get(paURL + paFilter + paParams)
dfApp = pd.DataFrame(response.json())
dfApp

,applicantName,city
0,Louisiana Department of Revenue,Baton Rouge
1,Southwest Louisiana Convention and Visitors B...,Lake Charles
2,"Iowa, Town of",Iowa


### Find Each Applicant in SAM.gov
For every applicant we call the SAM.gov ```entities``` endpoint, searching by legal business name and keeping only active registrations. From each match we pull the UEI, CAGE code, registration status, and expiration date.

<div class="alert alert-block alert-warning">
<strong>Requires an API Key:</strong> The SAM.gov API requires a free key (request one from your SAM.gov account under <em>Account Details &rarr; Public API Key</em>) and paste it into the <code>SAM_API_KEY</code> variable below. A brand-new personal key with no assigned account role is limited to <strong>10 requests per day</strong>; a key tied to an account role allows 1,000/day. Because each applicant is a separate request, keep the list short while testing. See the <a href="https://open.gsa.gov/api/entity-api/">Entity Management API documentation</a> for details.
</div>

In [8]:
# paste your free SAM.gov API key here (request one under Account Details -> Public API Key on SAM.gov)
SAM_API_KEY = "YOUR_KEY_HERE"
samURL = "https://api.sam.gov/entity-information/v3/entities"

def lookup_entity(name):
    """Search SAM.gov for an active registration matching a legal business name.
    Returns the entityRegistration dict for the first match, or None if there is no match."""
    params = {
        "api_key": SAM_API_KEY,
        "legalBusinessName": name,
        "registrationStatus": "A",              # 'A' = active registrations only
        "includeSections": "entityRegistration" # we only need the registration section
    }
    resp = requests.get(samURL, params=params, timeout=30)
    if resp.status_code == 429:
        raise RuntimeError("SAM.gov daily request limit reached for this key - try again tomorrow")
    resp.raise_for_status()
    data = resp.json()
    if data.get("totalRecords", 0) == 0:
        return None
    return data["entityData"][0]["entityRegistration"]

# look up each applicant and collect the registration details we care about
records = []
for name in dfApp["applicantName"]:
    reg = lookup_entity(name)
    records.append({
        "applicantName": name,
        "matchedName": reg["legalBusinessName"] if reg else None,
        "ueiSAM": reg["ueiSAM"] if reg else None,
        "cageCode": reg["cageCode"] if reg else None,
        "registrationStatus": reg["registrationStatus"] if reg else "Not found",
        "expirationDate": reg["registrationExpirationDate"] if reg else None
    })

dfSam = pd.DataFrame(records)
dfSam

,applicantName,matchedName,ueiSAM,cageCode,registrationStatus,expirationDate
0,Louisiana Department of Revenue,STATE OF LOUISIANA DEPARTMENT OF REVENUE,NF39ZNDBTL97,325F4,Active,2027-06-01
1,Southwest Louisiana Convention and Visitors B...,NaN,NaN,NaN,Not found,NaN
2,"Iowa, Town of",TOWN OF IOWA TOWN HALL,KSTKAJDHP2T9,6ZTK6,Active,2027-05-26


### Review the Enriched Result
Each FEMA applicant now carries its SAM.gov identity, the UEI (the identifier the federal government adopted in place of the old DUNS number), the CAGE code, and whether its registration is currently active. A test output looks like this:

| Applicant | UEI SAM | CAGE | Status | Expires |
|---|---|---|---|---|
| Louisiana Deptartment of Revenue | H8ABCDEF1234 | 7XY12 | Active | 2026-03-15 |
| Southwest Lousiana Convention and Visitors Bureau| — | — | Not found | — |
| Iowa, Town of | K2LMNOPQ5678 | 9AB34 | Active | 2025-12-01 |

<div class="alert alert-block alert-info">
<strong>Tip:</strong> The UEIs, CAGE codes, and dates come from live SAM.gov data. Therefore, the dates may be different in your results.
</div>

Notice that in the sample output, FEMA's applicant name ```Southwest Lousiana Convention and Visitors Bureau``` does not find a match in SAM.gov's system. This is because the applicant name differs from SAM.gov's ```SOUTHWEST LOUISIANA CONVENTION & VISITORS BUREAU```. A more sophisticated example would normalize the names before searching, join on a shared identifier such as the UEI, or use some kind of wildcard match.

## 4) USAspending.gov - Identifying FEMA Programs and Award Types
[**USAspending.gov**](https://api.usaspending.gov/) publishes every federal award and needs no API key. It can also provide general data , and needs no API key. Awards on USASpending.gov can be filtered by Program Number and Award Type Code. The API can return a list of these values rather than look on the website. For the following example the codes we will be identifing are:
* **Program number** - the Assistance Listing (formerly CFDA) number, for example ```97.036``` for Public Assistance.
* **Award type code** - the funding instrument, for example ```04``` for a project grant.

The following example filters spending data by the agency name and date. Program numbers and program titles are returned in order of award amount. Additionally, filtering by grants will return the various grant award type codes and descriptions.

In [3]:
# FEMA's assistance listings, ranked by dollars awarded over five federal fiscal years
catURL = "https://api.usaspending.gov/api/v2/search/spending_by_category/cfda/"
catPayload = {
    "filters": {
        "agencies": [{"type": "awarding", "tier": "subtier",
                      "name": "Federal Emergency Management Agency"}],
        "time_period": [{"start_date": "2019-10-01", "end_date": "2024-09-30"}]
    },
    "limit": 10, "page": 1
}
dfProgram = (pd.DataFrame(requests.post(catURL, json=catPayload).json()["results"])
             .rename(columns={"code": "programNumber", "name": "programTitle", "amount": "awarded"}))

# award type codes and descriptions, grouped by funding instrument
typeJson = requests.get("https://api.usaspending.gov/api/v2/references/award_types/").json()
dfAwardType = pd.DataFrame([(category, typeCode, description)
                            for category, typeCodes in typeJson.items()
                            for typeCode, description in typeCodes.items()],
                           columns=["category", "awardTypeCode", "description"])

display(dfProgram[["programNumber", "programTitle", "awarded"]])
display(dfAwardType[dfAwardType["category"] == "grants"])

,programNumber,programTitle,awarded
0,97.036,Disaster Grants - Public Assistance (President...,"$149,333,546,064.78"
1,97.050,Presidential Declared Disaster Assistance to I...,"$37,335,181,749.21"
2,97.048,Federal Disaster Assistance to Individuals and...,"$10,246,096,283.12"
3,97.022,Flood Insurance,"$10,186,025,618.60"
4,97.067,Homeland Security Grant Program,"$5,513,502,556.09"
5,97.039,Hazard Mitigation Grant,"$4,643,417,727.39"
6,97.042,Emergency Management Performance Grants,"$2,156,643,400.44"
7,97.024,Emergency Food and Shelter National Board Program,"$1,917,000,000.00"
8,97.044,Assistance to Firefighters Grant,"$1,892,513,905.48"
9,97.083,Staffing for Adequate Fire and Emergency Respo...,"$1,804,090,059.66"


,category,awardTypeCode,description
16,grants,02,Block Grant
17,grants,03,Formula Grant
18,grants,04,Project Grant
19,grants,05,Cooperative Agreement
20,grants,F001,Grant
21,grants,F002,Cooperative Agreement


## 5) USAspending.gov - PA Disaster Grants
Using the program number (CFDA) identified above, **97.036**, this example will list spending by award for some disasters in Louisiana. 

The award IDs are a constructed key, just like the Federal Register docket: ```<disasterNumber><declarationType><state>P<sequence>```, for example ```4559DRLAP00000001```. Parsing the front of it hands back a FEMA disaster number.

<div class="alert alert-block alert-warning">
<strong>NOTE:</strong> A comparision between USAspending and OpenFEMA Public Assistance data will yield different results. Data comes from two different systems and we (OpenFEMA) do not know the underlying criterial that USAspending uses to aggregate data.
</div>


In [6]:
# every CFDA 97.036 (Disaster Grants - Public Assistance) grant performed in Louisiana
usaURL = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
usaPayload = {
    "filters": {
        "award_type_codes": ["02", "03", "04", "05"],   # grant award types
        "program_numbers": ["97.036"],                  # Disaster Grants - Public Assistance
        "time_period": [{"start_date": "2019-01-01", "end_date": "2023-09-30"}],
        "place_of_performance_locations": [{"country": "USA", "state": "LA"}]
    },
    "fields": ["Award ID", "Recipient Name", "Award Amount", "Start Date"],
    "sort": "Award Amount", "order": "desc", "limit": 10, "page": 1
}
dfAward = pd.DataFrame(requests.post(usaURL, json=usaPayload).json()["results"])

# pull the disaster number and declaration type back out of the award ID
dfAward[["disasterNumber", "declarationType"]] = dfAward["Award ID"].str.extract(r"^(\d{4,5})(DR|EM)")
dfAward = dfAward.dropna(subset=["disasterNumber"]).astype({"disasterNumber": int})

dfAward[["Award ID", "disasterNumber", "declarationType", "Recipient Name", "Award Amount", "Start Date"]]

,Award ID,disasterNumber,declarationType,Recipient Name,Award Amount,Start Date
0,4611DRLAP00000001,4611,DR,HOMELAND SECURITY & EMERGENCY,"$3,623,924,677.05",2021-09-17
1,4559DRLAP00000001,4559,DR,HOMELAND SECURITY & EMERGENCY,"$3,209,955,430.85",2020-10-02
2,1603DRLAP0000001,1603,DR,HOMELAND SECURITY & EMERGENCY,"$2,390,215,653.15",2005-09-07
3,4484DRLAP00000001,4484,DR,HOMELAND SECURITY & EMERGENCY,"$2,296,882,329.99",2020-03-26
4,4277DRLAP0000001,4277,DR,HOMELAND SECURITY & EMERGENCY,"$749,147,429.40",2016-08-19
5,4577DRLAP00000001,4577,DR,HOMELAND SECURITY & EMERGENCY,"$134,709,548.06",2021-03-05
6,4570DRLAP00000001,4570,DR,HOMELAND SECURITY & EMERGENCY,"$123,341,559.72",2020-12-14
7,4263DRLAP0000001,4263,DR,HOMELAND SECURITY & EMERGENCY,"$104,688,013.59",2016-03-17
8,4458DRLAP00000001,4458,DR,HOMELAND SECURITY & EMERGENCY,"$49,410,343.11",2019-11-15
9,4590DRLAP00000001,4590,DR,HOMELAND SECURITY & EMERGENCY,"$36,916,838.29",2021-05-14


Similar lookups can be done for Hazard Mitigation Grants.

## 6) Federal Register - Disaster Notices and Amendments
Every major disaster and emergency declaration and subsequent amendments are published as an official notice in the *Federal Register*. Given a FEMA disaster number, this example retrieves the disaster's original declaration notice plus every amendment. The [**Federal Register API**](https://www.federalregister.gov/developers/documentation/api/v1) is free and requires no API key.

<div class="alert alert-block alert-warning">
<strong>Warning:</strong> Although the FEMA Federal Register Notices page reflects the same amendment data, there is a timing difference. FEMA receives and processes amendment notifications prior to their inclusion in the Federal Register API.
</div>

### Build the Docket Number and Query the API
Our shared key this time is constructed rather than normalized/transformed. FEMA files each disaster's notices under an *internal agency docket* in the following form: ```FEMA-<disasterNumber>-<declarationType>```. For example, ```FEMA-4699-DR``` for major disaster 4699. 

We retrieve the disaster from the OpenFEMA ```DisasterDeclarationsSummaries``` dataset, and use the syntax above to assemble the docket number. This key is then applied against the Federal Register API as a ```conditions[docket_id]``` filter.

In [11]:
# the FEMA disaster we want the Federal Register history for
disasterNumber = 4699

# get the declaration from OpenFEMA
# Federal Register docket is FEMA-<number>-<type>
declURL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries?"
declFilter = "$filter=disasterNumber%20eq%20" + str(disasterNumber)
declParams = "&$select=disasterNumber,state,declarationType&$top=1&$format=jsona&$metadata=off"
declaration = requests.get(declURL + declFilter + declParams).json()[0]

# build the shared key
docket = "FEMA-" + str(declaration["disasterNumber"]) + "-" + declaration["declarationType"]
print("Disaster " + str(disasterNumber) + " (" + declaration["state"] + ") -> docket " + docket)

# ask the Federal Register API for every document filed under that docket
frURL = "https://www.federalregister.gov/api/v1/documents.json"
frParams = {
    "conditions[docket_id]": docket,
    "order": "oldest",
    "per_page": 100,
    "fields[]": ["publication_date", "title", "document_number", "html_url"]
}
response = requests.get(frURL, params=frParams)
frJson = response.json()

dfFR = pd.DataFrame(frJson["results"])
print(str(frJson["count"]) + " Federal Register documents filed under " + docket)

# pandas truncates cell values longer than 50 characters when displaying, which cuts off the html_url links
with pd.option_context("display.max_colwidth", None):
    display(dfFR)

Disaster 4699 (CA) -> docket FEMA-4699-DR
10 Federal Register documents filed under FEMA-4699-DR


,publication_date,title,document_number,html_url
0,2023-05-31,California; Major Disaster and Related Determinations,2023-11529,https://www.federalregister.gov/documents/2023/05/31/2023-11529/california-major-disaster-and-related-determinations
1,2023-05-31,California; Amendment No. 1 to Notice of a Major Disaster Declaration,2023-11530,https://www.federalregister.gov/documents/2023/05/31/2023-11530/california-amendment-no-1-to-notice-of-a-major-disaster-declaration
2,2023-05-31,California; Amendment No. 2 to Notice of a Major Disaster Declaration,2023-11531,https://www.federalregister.gov/documents/2023/05/31/2023-11531/california-amendment-no-2-to-notice-of-a-major-disaster-declaration
3,2023-08-02,California; Amendment No. 3 to Notice of a Major Disaster Declaration,2023-16329,https://www.federalregister.gov/documents/2023/08/02/2023-16329/california-amendment-no-3-to-notice-of-a-major-disaster-declaration
4,2023-08-02,California; Amendment No. 4 to Notice of a Major Disaster Declaration,2023-16330,https://www.federalregister.gov/documents/2023/08/02/2023-16330/california-amendment-no-4-to-notice-of-a-major-disaster-declaration
5,2023-08-02,California; Amendment No. 5 to Notice of a Major Disaster Declaration,2023-16331,https://www.federalregister.gov/documents/2023/08/02/2023-16331/california-amendment-no-5-to-notice-of-a-major-disaster-declaration
6,2023-09-27,California; Amendment No. 6 to Notice of a Major Disaster Declaration,2023-21003,https://www.federalregister.gov/documents/2023/09/27/2023-21003/california-amendment-no-6-to-notice-of-a-major-disaster-declaration
7,2023-09-27,California; Amendment No. 7 to Notice of a Major Disaster Declaration,2023-21004,https://www.federalregister.gov/documents/2023/09/27/2023-21004/california-amendment-no-7-to-notice-of-a-major-disaster-declaration
8,2023-10-20,California; Amendment No. 8 to Notice of a Major Disaster Declaration,2023-23184,https://www.federalregister.gov/documents/2023/10/20/2023-23184/california-amendment-no-8-to-notice-of-a-major-disaster-declaration
9,2023-11-30,California; Amendment No. 9 to Notice of a Major Disaster Declaration,2023-26273,https://www.federalregister.gov/documents/2023/11/30/2023-26273/california-amendment-no-9-to-notice-of-a-major-disaster-declaration


The above example only returns a few fields selected from the Federal Register API. A simple call using the document number (not the Docket Number) will return a detailed JSON object with many other attributes. Among them are links to other resources such as the raw text of the notice and a PDF document. 

In [21]:
import json

# ask the Federal Register API the full object filed under the third amendment above
frURL = "https://www.federalregister.gov/api/v1/documents/2023-16329.json"

response = requests.get(frURL)
frJson = response.json()
print(json.dumps(frJson, indent=4))

{
    "abstract": "This notice amends the notice of a major disaster declaration for the State of California (FEMA-4699-DR), dated April 3, 2023, and related determinations.",
    "action": "Notice.",
    "agencies": [
        {
            "raw_name": "DEPARTMENT OF HOMELAND SECURITY",
            "name": "Homeland Security Department",
            "id": 227,
            "url": "https://www.federalregister.gov/agencies/homeland-security-department",
            "json_url": "https://www.federalregister.gov/api/v1/agencies/227",
            "parent_id": null,
            "slug": "homeland-security-department"
        },
        {
            "raw_name": "FEDERAL EMERGENCY MANAGEMENT AGENCY",
            "name": "Federal Emergency Management Agency",
            "id": 166,
            "url": "https://www.federalregister.gov/agencies/federal-emergency-management-agency",
            "json_url": "https://www.federalregister.gov/api/v1/agencies/166",
            "parent_id": 227,
          

## 7) Federal Register - Declarations in a Date Range, Enriched with FEMA Data
This example pulls data from the Federal Register first. It attempts to answer the question **"What major disasters were declared for a given date range, and what monies has FEMA spent?"**

### Retrieve the Declaration Notices and Parse Out the Key
Every initial declaration notice for a major disaster shares one title format: ```<State>; Major Disaster and Related Determinations``` so an exact-phrase search for that title returns one document per new major disaster.

The search response has no disaster-number field, but each document's ```docket_ids``` list contains the same internal agency docket Example 1 built. So we run Example 1's construction *backward*, parsed from the federal register.

In [11]:
# Our publication date range
startDate = "2023-01-01"
endDate = "2023-06-30"

# search for the initial declaration notices limited to FEMA
frURL = "https://www.federalregister.gov/api/v1/documents.json"
frParams = {
    "conditions[agencies][]": "federal-emergency-management-agency",
    "conditions[term]": "\"Major Disaster and Related Determinations\"",
    "conditions[publication_date][gte]": startDate,
    "conditions[publication_date][lte]": endDate,
    "order": "oldest",
    "per_page": 100,   # a wider date range may exceed one page
    "fields[]": ["publication_date", "title", "document_number", "docket_ids"]
}
frJson = requests.get(frURL, params=frParams).json()
dfDecl = pd.DataFrame(frJson["results"])
print(str(frJson["count"]) + " major disaster declarations published between " + startDate + " and " + endDate)

# Parse disaster number from docket ID
def extract_disaster_number(docket_ids):
    for docket in docket_ids:
        match = re.search(r"FEMA-(\d+)-DR", docket)
        if match:
            return int(match.group(1))
    return None

dfDecl["disasterNumber"] = dfDecl["docket_ids"].apply(extract_disaster_number)
dfDecl[["publication_date", "title", "disasterNumber"]].head()

33 major disaster declarations published between 2023-01-01 and 2023-06-30


,publication_date,title,disasterNumber
0,2023-01-09,South Carolina; Major Disaster and Related Det...,4677
1,2023-01-09,West Virginia; Major Disaster and Related Dete...,4678
2,2023-01-09,West Virginia; Major Disaster and Related Dete...,4679
3,2023-01-31,Florida; Major Disaster and Related Determinat...,4680
4,2023-03-02,Alabama; Major Disaster and Related Determinat...,4684


### Enrich the Declarations with OpenFEMA Data
With a ```disasterNumber``` parsed for each row, we can now call OpenFEMA's API. OpenFEMA's ```in``` operator accepts the whole list of disaster numbers in a single call. We do this twice.

- **DisasterDeclarationsSummaries** describes each disaster (incident type, declaration title). It holds one row per designated area.
- **FemaWebDisasterSummaries** carries the financial rollups, total Individuals and Households Program (IHP) dollars approved and total Public Assistance (PA) dollars obligated, one row per disaster.

Joining both back to the Federal Register list on the parsed key produces a single table with every major disaster declared in the window, when it was published, what happened, how many areas were designated, and what FEMA has spent so far.

In [12]:
# build the filter value once, OpenFEMA's 'in' operator takes the whole set
numberList = ",".join(str(n) for n in dfDecl["disasterNumber"])

# one row per designated area, so group down to one row per disaster
declURL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
declParams = {
    "$filter": "disasterNumber in (" + numberList + ")",
    "$select": "disasterNumber,incidentType,declarationTitle,designatedArea",
    "$format": "jsona", "$metadata": "off", "$allrecords": "true"
}
dfAreas = pd.DataFrame(requests.get(declURL, params=declParams).json())
dfInfo = dfAreas.groupby("disasterNumber").agg(
    incidentType=("incidentType", "first"),
    declarationTitle=("declarationTitle", "first"),
    areasDesignated=("designatedArea", "nunique"))

# already one row per disaster
sumURL = "https://www.fema.gov/api/open/v1/FemaWebDisasterSummaries"
sumParams = {
    "$filter": "disasterNumber in (" + numberList + ")",
    "$select": "disasterNumber,totalAmountIhpApproved,totalObligatedAmountPa",
    "$format": "jsona", "$metadata": "off"
}
dfDollars = pd.DataFrame(requests.get(sumURL, params=sumParams).json()).set_index("disasterNumber")

# join both OpenFEMA tables back to the Federal Register list on the parsed key
dfCombined = (dfDecl[["publication_date", "title", "disasterNumber"]]
              .join(dfInfo, on="disasterNumber")
              .join(dfDollars, on="disasterNumber")
              .sort_values("publication_date"))

with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(dfCombined)

,publication_date,title,disasterNumber,incidentType,declarationTitle,areasDesignated,totalAmountIhpApproved,totalObligatedAmountPa
0,2023-01-09,South Carolina; Major Disaster and Related Determinations,4677,Hurricane,HURRICANE IAN,46,"$3,383,846.20","$21,782,280.51"
1,2023-01-09,West Virginia; Major Disaster and Related Determinations,4678,Flood,"SEVERE STORMS, FLOODING, LANDSLIDES, AND MUDSLIDES",1,NaN,"$3,082,243.70"
2,2023-01-09,West Virginia; Major Disaster and Related Determinations,4679,Severe Storm,"SEVERE STORMS, FLOODING, LANDSLIDES, AND MUDSLIDES",1,NaN,"$3,942,957.39"
3,2023-01-31,Florida; Major Disaster and Related Determinations,4680,Hurricane,HURRICANE NICOLE,63,"$7,016,325.08","$126,744,321.83"
4,2023-03-02,Alabama; Major Disaster and Related Determinations,4684,Severe Storm,"SEVERE STORMS, STRAIGHT-LINE WINDS, AND TORNADOES",13,"$9,492,928.94","$16,396,912.59"
5,2023-03-02,Georgia; Major Disaster and Related Determinations,4685,Severe Storm,"SEVERE STORMS, STRAIGHT-LINE WINDS, AND TORNADOES",9,"$5,557,341.55","$36,218,174.64"
6,2023-03-02,Havasupai Tribe; Major Disaster and Related Determinations,4681,Flood,FLOODING,1,NaN,"$1,047,825.39"
7,2023-03-02,Washington; Major Disaster and Related Determinations,4682,Severe Storm,"SEVERE WINTER STORM, STRAIGHT-LINE WINDS, FLOODING, LANDSLIDES, AND MUDSLIDES",9,NaN,"$17,919,129.86"
8,2023-03-02,California; Major Disaster and Related Determinations,4683,Flood,"SEVERE WINTER STORMS, FLOODING, LANDSLIDES, AND MUDSLIDES",44,"$61,173,467.31","$403,867,880.90"
15,2023-04-13,Rosebud Sioux Tribe; Major Disaster and Related Determinations,4687,Winter Storm,SEVERE WINTER STORMS AND SNOWSTORM,1,NaN,"$819,070.62"


One caution about reading the table, not every ```NaN``` is a failed join! These disasters show no IHP dollars correctly.


## 8) Weather.gov - IPAWS Alerts Issued for a UGC

The National Weather Service (NWS) identifies the areas utilized in forcasts, advisories, etc., by **Universal Geographic Codes (UGCs)**. A UGC names either a *county* or an NWS-drawn *zone* such as a public forecast zone or fire weather zone. If you want to connect FEMA disaster data to the weather hazards, the UGC is the key you need. The NWS geolocation interface is described in the <a href="https://www.weather.gov/media/documentation/docs/NWS_Geolocation.pdf">NWS Geolocation documentation (PDF)</a>.

<div class="alert alert-block alert-warning">
    <strong>API Use:</strong> Like the Federal Register API, the <strong>weather.gov API</strong> is free and requires no API key, but it does ask that every client identify itself with a <code>User-Agent</code> header for each HTTP request.
</div>

Given a latitude/longitude, one call to the ```/zones``` endpoint with a ```point``` parameter returns every NWS zone containing that location. Each zone's ```id``` is its UGC. Here we look up Lake Charles, Louisiana, the city at the center of Hurricane Laura's landfall in the Census and SAM.gov examples. If you start from a place name instead of coordinates, geocode it first.

The county UGC is also a composite key - a two-letter state abbreviation followed by the three FIPS digits, which is exactly what OpenFEMA's ```state``` and ```fipsCountyCode``` fields hold. The next example puts it to work.

In [6]:
# the location we want UGC codes for, Lake Charles, LA
lat = 30.2266
lon = -93.2174

# as requested in the API documentation, pass in a user-agent for identification
nwsHeaders = {"User-Agent": "openfema-samples tutorial (openfema@fema.dhs.gov)"}

# one call to the zones endpoint returns every NWS zone that contains the point
nwsURL = "https://api.weather.gov/zones"
nwsParams = {
    "point": str(lat) + "," + str(lon),
    "include_geometry": "false"
}
response = requests.get(nwsURL, params=nwsParams, headers=nwsHeaders)
zonesJson = response.json()

# response is GeoJSON, one feature per zone, feature's 'id' is UGC
dfUGC = pd.DataFrame([{
    "ugc": f["properties"]["id"],
    "type": f["properties"]["type"],
    "name": f["properties"]["name"],
    "state": f["properties"]["state"]
} for f in zonesJson["features"]])

print(str(len(dfUGC)) + " zones contain the point (" + str(lat) + ", " + str(lon) + ")")
dfUGC

3 zones contain the point (30.2266, -93.2174)


,ugc,type,name,state
0,LAC019,county,Calcasieu,LA
1,LAZ241,fire,Southern Calcasieu,LA
2,LAZ241,public,Southern Calcasieu,LA


### Using a found UGS to retrieve IPAWS alerts
The IPAWS Archived Alerts dataset contains UGC as an event identifier. For a given UGC, we can retrieve alerts by UGC - the majority will be meteorological from the NWS. The UGC is located in the ```info/area/geocode``` array element block of the alert. We will use the OpenFEMA ```contains()``` operator as part of a filter.


In [16]:
# the county UGC from the point lookup is the key IPAWS stores in info/area/geocode
countyUGC = dfUGC[dfUGC["type"] == "county"].iloc[0]["ugc"]

ipawsURL = "https://www.fema.gov/api/open/v1/IpawsArchivedAlerts"
ipawsParams = {
    "$filter": "sent gt '2020-08-25T00:00:00.000z' and sent lt '2020-08-29T00:00:00.000z'"
               " and contains(info/area/geocode/value,'" + countyUGC + "')"
               " and contains(info/category,'Met')",
    "$select": "identifier,sent,info", "$top": "100", "$format": "jsona", "$metadata": "off"
}
alertJson = requests.get(ipawsURL, params=ipawsParams, timeout=300).json()

# info repeats once per language, take the first
dfAlerts = pd.DataFrame([{
    "identifier": a["identifier"],
    "sent": a["sent"][:19],
    "event": a["info"][0]["event"],
    "severity": a["info"][0]["severity"],
    "sender": a["info"][0]["senderName"]
} for a in alertJson]).sort_values("sent")

print(str(len(dfAlerts)) + " meteorological IPAWS alerts for " + countyUGC + " during Hurricane Laura's landfall")
dfAlerts.head(10)

47 meteorological IPAWS alerts for LAC019 during Hurricane Laura's landfall


,identifier,sent,event,severity,sender
0,NWS-IDP-PROD-4410376-3676040,2020-08-25T16:00:00,Flood Warning,Severe,NWS Lake Charles LA
1,NWS-IDP-PROD-4410375-3676039,2020-08-25T16:00:00,Flood Warning,Severe,NWS Lake Charles LA
2,NWS-IDP-PROD-4410373-3676037,2020-08-25T16:00:00,Flood Warning,Severe,NWS Lake Charles LA
3,NWS-IDP-PROD-4410729-3676262,2020-08-25T19:49:00,Flood Warning,Severe,NWS Lake Charles LA
4,NWS-IDP-PROD-4412031-3677258,2020-08-26T04:08:00,Flood Warning,Severe,NWS Lake Charles LA
9,NWS-IDP-PROD-4412856-3677847,2020-08-26T17:30:00,Severe Thunderstorm Warning,Severe,NWS Lake Charles LA
5,NWS-IDP-PROD-4412873-3677857,2020-08-26T17:41:00,Flood Warning,Severe,NWS Lake Charles LA
6,NWS-IDP-PROD-4412875-3677858,2020-08-26T17:44:00,Flood Warning,Severe,NWS Lake Charles LA
7,NWS-IDP-PROD-4412878-3677859,2020-08-26T17:49:00,Flood Warning,Severe,NWS Lake Charles LA
10,NWS-IDP-PROD-4412931-3677906,2020-08-26T18:03:00,Severe Thunderstorm Warning,Severe,NWS Lake Charles LA


## 9) Weather.gov - Retrieving an NWS Alert from an IPAWS Alert
IPAWS records that originated with the National Weather Service carry the alert's own identifier with a ```.IPAWS``` suffix appended. Strip the suffix and the value is the key for the weather.gov ```/alerts/{id}``` endpoint, which returns the alert as NWS serves it - parsed fields, affected zones, and the full instruction text. This may not be very useful as the IPAWS alert contains most of (if not all) the original NWS alert.

<div class="alert alert-block alert-warning">
<strong>Warning:</strong> The weather.gov alerts endpoint only serves roughly the last <strong>7 days</strong>. Any older identifier returns a 404, so this lookup cannot be used to research alerts tied to a past disaster. For historical weather see the NCEI example below.
</div>

In [17]:
# grab a recent IPAWS record that came from the NWS
recentCutoff = (datetime.now(timezone.utc) - timedelta(days=5)).strftime("%Y-%m-%dT%H:%M:%S.000z")
recentParams = {"$filter": "sent gt '" + recentCutoff + "'", "$select": "identifier,sent",
                "$orderby": "sent desc", "$top": "500", "$format": "jsona", "$metadata": "off"}
nwsSourced = [a for a in requests.get(ipawsURL, params=recentParams).json()
              if a["identifier"].endswith(".IPAWS")]

# strip the suffix to get the weather.gov alert id
alertId = nwsSourced[0]["identifier"].removesuffix(".IPAWS")
print(nwsSourced[0]["identifier"] + "\n  -> " + alertId)

nwsAlert = requests.get("https://api.weather.gov/alerts/" + alertId, headers=nwsHeaders).json()["properties"]
pd.Series({k: nwsAlert[k] for k in ["event", "severity", "certainty", "urgency", "areaDesc",
                                    "effective", "expires", "senderName", "headline"]})

urn:oid:2.49.0.1.840.0.83aa327d4e95089cdee03d97da3c9105fb7d90e1.001.1.IPAWS
  -> urn:oid:2.49.0.1.840.0.83aa327d4e95089cdee03d97da3c9105fb7d90e1.001.1


event                                       Flash Flood Warning
severity                                                 Severe
certainty                                                Likely
urgency                                               Immediate
areaDesc                                              Pecos, TX
effective                             2026-09-26T15:12:00-05:00
expires                               2026-09-26T17:15:00-05:00
senderName                                NWS Midland/Odessa TX
headline      Flash Flood Warning issued September 26 at 3:1...
dtype: str

## 10) NCEI - Historical Weather Events by Disaster Declaration
The weather.gov alerts endpoint is a live feed, so historical research requires a different source. NOAA's National Centers for Environmental Information (NCEI) maintains the [**Storm Events Database**](https://www.ncdc.noaa.gov/stormevents/), the official record of severe weather back to 1950 including tornadoes, hail, wind, floods, plus deaths, injuries, and damage estimates.

<div class="alert alert-block alert-info">
<strong>Note:</strong> The NCEI does have an API (https://www.ncei.noaa.gov/support/access-data-service-api-user-documentation), however downloading bulk storm events are more useful for this example. NCEI's does not maintain files for individual storm events, instead publishing one gzipped CSV per year. 
</div>

<div class="alert alert-block alert-warning">
<strong>Warning:</strong> The NCEI Storm Events Database does not directly link back to a disaster declaration. This example returns those Storm Event records that match a declared disaster's incident begin and end date, and the county in which the event occured. It could further be filtered by incident type, but often many types of events occur (e.g., hail, tornado, rain) within a declared disaster.
</div>

### Retrieve the Declaration and the Storm Events File

In [18]:
# DR-4777, the March 2024 Ohio tornado outbreak
disasterNumber = 4777
declURL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
declParams = {"$filter": "disasterNumber eq " + str(disasterNumber),
              "$select": "fipsStateCode,fipsCountyCode,designatedArea,incidentBeginDate,incidentEndDate",
              "$format": "jsona", "$metadata": "off"}
dfDeclared = pd.DataFrame(requests.get(declURL, params=declParams).json())
incidentBegin = dfDeclared["incidentBeginDate"].min()[:10]
incidentEnd = dfDeclared["incidentEndDate"].max()[:10]
print(str(len(dfDeclared)) + " counties designated, incident " + incidentBegin + " to " + incidentEnd)

# the file name carries a 'created' date that changes whenever NCEI revises the year, so look it up
stormBase = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
fileName = re.findall(r"StormEvents_details-ftp_v1\.0_d" + incidentBegin[:4] + r"_c\d+\.csv\.gz",
                      requests.get(stormBase).text)[0]
dfStorm = pd.read_csv(stormBase + fileName, compression="gzip", low_memory=False)
print(str(len(dfStorm)) + " storm events nationwide in " + incidentBegin[:4] + " (" + fileName + ")")

11 counties designated, incident 2024-03-14 to 2024-03-14
69801 storm events nationwide in 2024 (StormEvents_details-ftp_v1.0_d2024_c20260728.csv.gz)


### Join the Events to the Declared Counties
Each event in the NCEI data row contains a ```STATE_FIPS``` and ```CZ_FIPS```, the same FIPS pair the Census example joined on. This permits a two-part join to a declaration's designated counties, bounded by the incident period.
```CZ_TYPE``` tells us how a row is located. ```C``` rows are county-based and their ```CZ_FIPS``` is a county FIPS code; ```Z``` rows are located by NWS forecast zone and their ```CZ_FIPS``` is a zone number, not a FIPS code. Only the ```C``` rows can join, which is fine for a tornado outbreak but would drop most of a hurricane or winter storm.

The resulting weather events are likely related to the FEMA disaster declaration, but not necessarily included in the official disaster declaration.

In [19]:
# keep county-located rows and pad the FIPS codes to match OpenFEMA's
dfCounty = dfStorm[dfStorm["CZ_TYPE"] == "C"].copy()
dfCounty["fipsStateCode"] = dfCounty["STATE_FIPS"].astype(str).str.zfill(2)
dfCounty["fipsCountyCode"] = dfCounty["CZ_FIPS"].astype(str).str.zfill(3)
dfCounty["beginDate"] = pd.to_datetime(dfCounty["BEGIN_DATE_TIME"], format="%d-%b-%y %H:%M:%S")

# bound by the incident period, then inner join to the declared counties
inWindow = dfCounty[(dfCounty["beginDate"] >= incidentBegin) &
                    (dfCounty["beginDate"] < pd.Timestamp(incidentEnd) + pd.Timedelta(days=1))]
dfEvents = inWindow.merge(dfDeclared[["fipsStateCode", "fipsCountyCode", "designatedArea"]],
                          on=["fipsStateCode", "fipsCountyCode"])

print(str(len(inWindow)) + " US events in the incident window, " +
      str(len(dfEvents)) + " of them inside the declared counties")
dfEvents[["designatedArea", "EVENT_TYPE", "BEGIN_DATE_TIME", "TOR_F_SCALE", "DEATHS_DIRECT",
          "INJURIES_DIRECT", "DAMAGE_PROPERTY"]].sort_values("BEGIN_DATE_TIME")

553 US events in the incident window, 27 of them inside the declared counties


,designatedArea,EVENT_TYPE,BEGIN_DATE_TIME,TOR_F_SCALE,DEATHS_DIRECT,INJURIES_DIRECT,DAMAGE_PROPERTY
0,Mercer (County),Hail,14-MAR-24 17:42:00,NaN,0,0,0.00K
17,Mercer (County),Hail,14-MAR-24 17:43:00,NaN,0,0,1.00K
24,Mercer (County),Tornado,14-MAR-24 17:45:00,EF1,0,0,250.00K
16,Mercer (County),Tornado,14-MAR-24 18:00:00,EF1,0,0,500.00K
23,Auglaize (County),Tornado,14-MAR-24 18:08:00,EF1,0,0,250.00K
11,Hancock (County),Tornado,14-MAR-24 18:09:00,EFU,0,0,0.00K
12,Hancock (County),Hail,14-MAR-24 18:29:00,NaN,0,0,0.00K
15,Auglaize (County),Tornado,14-MAR-24 18:29:00,EF3,0,2,1.00M
10,Hancock (County),Tornado,14-MAR-24 18:30:00,EF1,0,0,75.00K
25,Logan (County),Tornado,14-MAR-24 18:43:00,EF3,3,25,5.00M


## 11) USGS - Earthquake Events by Disaster Declaration
The [**USGS FDSN event API**](https://earthquake.usgs.gov/fdsnws/event/1/) serves the global earthquake catalog and requires no API key. It permits filtering by time and a coordinate bounding box. The incident period of a FEMA earthquake declaration supplies the time window, while its designated areas provides a way to bound the area.

The FEMA Disaster Declarations Summaries dataset does not include county geometry, so we get the bounding box from **Census TIGERweb**, which accepts the same state and county FIPS codes the declaration carries.

<div class="alert alert-block alert-info">
<strong>Note:</strong> This example was created to both show the translation of counties into coordinates as well as the retrieval of earthquake data. An easier method is to use the Fema Web Disaster Declarations dataset. This dataset contains a link to a GeoJson file (geoJsonUrl) that will return the FEMA defined bounding box for a disaster - bypassing the Census step.
</div>

<div class="alert alert-block alert-info">
<strong>Note:</strong> Another interesting example would be to correlate seismic events with IPAWS Archived Alerts.
</div>

<div class="alert alert-block alert-warning">
<strong>Warning:</strong> As is the case with the NCEI Storm Events data, the USGS earthquake data does not directly link back to a disaster declaration. This example returns earthquake event records that match a declared disaster's incident begin and end date, and the county in which the event occured. There may be siesmic events that occured around the same time and location that did not warrent an official disaster declaration.
</div>

### Declaration to Bounding Box

In [20]:
# the OpenFEMA side: DR-4548, the March 2020 Magna, Utah earthquake and aftershocks
declURL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
quakeParams = {"$filter": "disasterNumber eq 4548",
               "$select": "fipsStateCode,fipsCountyCode,designatedArea,incidentBeginDate,incidentEndDate",
               "$format": "jsona", "$metadata": "off"}
dfQuakeDecl = pd.DataFrame(requests.get(declURL, params=quakeParams).json())

# Census TIGERweb returns the extent of the selected counties, in lat/lon when outSR is 4326
countyList = ",".join("'" + c + "'" for c in dfQuakeDecl["fipsCountyCode"])
tigerURL = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/1/query"
tigerParams = {"where": "STATE='" + dfQuakeDecl["fipsStateCode"][0] + "' AND COUNTY IN (" + countyList + ")",
               "returnExtentOnly": "true", "outSR": "4326", "f": "json"}
box = requests.get(tigerURL, params=tigerParams).json()["extent"]

print(", ".join(dfQuakeDecl["designatedArea"]))
print(box)

Davis (County), Salt Lake (County)
{'xmin': -112.49351499959144, 'ymin': 40.41416900006225, 'xmax': -111.5531879997636, 'ymax': 41.15438399976757, 'spatialReference': {'wkid': 4326, 'latestWkid': 4326}}


### Retrieve the Earthquake Data
The incident period and the bounding box represent the whole join. Each USGS earthquake event returns data that FEMA does not track: ```felt```, the count of "Did You Feel It?" reports, and ```mmi```, the ShakeMap instrumental intensity.

In [21]:
usgsParams = {
    "format": "geojson",
    "starttime": dfQuakeDecl["incidentBeginDate"].min()[:10],
    "endtime": dfQuakeDecl["incidentEndDate"].max()[:10],
    "minlatitude": box["ymin"], "maxlatitude": box["ymax"],
    "minlongitude": box["xmin"], "maxlongitude": box["xmax"],
    "minmagnitude": 3.5, "orderby": "magnitude"
}
quakeJson = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query", params=usgsParams).json()

dfQuakes = pd.DataFrame([{
    "utcTime": datetime.fromtimestamp(f["properties"]["time"] / 1000, timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "magnitude": f["properties"]["mag"],
    "place": f["properties"]["place"],
    "feltReports": f["properties"]["felt"],   # 'Did You Feel It?' responses
    "maxIntensity": f["properties"]["mmi"]    # ShakeMap instrumental intensity
} for f in quakeJson["features"]])

print(str(quakeJson["metadata"]["count"]) + " M3.5+ events inside the declared counties during the incident period")
dfQuakes

18 M3.5+ events inside the declared counties during the incident period


,utcTime,magnitude,place,feltReports,maxIntensity
0,2020-03-18 13:09,5.70,"5 km NNE of Magna, Utah",26376,6.925
1,2020-03-18 14:02,4.59,"6 km NNE of Magna, Utah",1406,4.650
2,2020-03-18 19:12,4.57,"5 km NE of Magna, Utah",3147,5.590
3,2020-04-16 13:41,4.19,"4 km NE of Magna, Utah",6730,4.920
4,2020-03-18 13:10,4.17,"1 km NNW of Magna, Utah",1,4.310
5,2020-04-15 02:56,4.17,"4 km NE of Magna, Utah",7104,4.780
6,2020-03-18 13:10,4.01,"4 km NNE of Magna, Utah",13,4.350
7,2020-03-18 13:10,3.94,"4 km NNE of Magna, Utah",33,3.980
8,2020-03-23 01:17,3.94,"8 km NNE of Magna, Utah",4434,4.730
9,2020-03-18 13:32,3.86,"6 km NE of Magna, Utah",517,4.590


## 12) USGS - Water Level by Disaster and County
The [**USGS Water Data APIs**](https://api.waterdata.usgs.gov/) delivers data from the National Stream Gauge Network as a RESTful, Open Geospatial Consortium (OGC) compliant API. An API key is not required. As we demonstrated in the Earthquake and Weather data, we can idenitfy stream guage data by a declared disaster's incident period and declared counties. We will use two USGS water data features here:

- ```monitoring-locations``` - the gauges, each tagged with ```state_code``` and ```county_code```, the same FIPS pair OpenFEMA uses
- ```daily``` - daily values per gauge, filtered by parameter (```00065``` is gage height) and statistic (```00003``` is the daily mean)

<div class="alert alert-block alert-info">
<strong>Note:</strong> Stream gauge height is measured relative to a gauge's site height. Raw numbers are not comparable between sites. The **rise**, or peak stage minus the stage at the start of the incident is what is relavant. That is what this example reports.
</div>


### Declared County Stream Gauges

In [ ]:
# the OpenFEMA side: DR-4906, the December 2025 flooding in Washington State
declURL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
floodParams = {"$filter": "disasterNumber eq 4906",
               "$select": "fipsStateCode,fipsCountyCode,designatedArea,incidentBeginDate,incidentEndDate",
               "$format": "jsona", "$metadata": "off"}
dfFlood = pd.DataFrame(requests.get(declURL, params=floodParams).json()).drop_duplicates("fipsCountyCode")
floodBegin = dfFlood["incidentBeginDate"].min()[:10]
floodEnd = dfFlood["incidentEndDate"].max()[:10]

waterBase = "https://api.waterdata.usgs.gov/ogcapi/v1/collections/"
countyIn = ",".join("'" + c + "'" for c in sorted(dfFlood["fipsCountyCode"]))
whereCounties = ("state_code='" + dfFlood["fipsStateCode"].iloc[0] + "'"
                 " AND county_code IN (" + countyIn + ") AND site_type_code='ST'")

locJson = requests.get(waterBase + "monitoring-locations/items",
                       params={"filter": whereCounties, "limit": 5000, "f": "json"}).json()
dfGauges = pd.DataFrame([{"monitoring_location_id": f["properties"]["id"],
                          "gauge": f["properties"]["monitoring_location_name"],
                          "fipsCountyCode": f["properties"]["county_code"]} for f in locJson["features"]])

print(str(len(dfFlood)) + " counties designated (" + floodBegin + " to " + floodEnd + "), " +
      str(len(dfGauges)) + " stream gauges in them")
dfGauges.head()

25 counties designated (2025-12-05 to 2025-12-19), 3640 stream gauges in them


,monitoring_location_id,gauge,fipsCountyCode
0,USGS-12009400,"BEAR BRANCH AT RM 4.8 NEAR CHINOOK, WA",049
1,USGS-12009450,"BEAR BRANCH AT RM 3.2 NEAR CHINOOK, WA",049
2,USGS-12009500,"BEAR BRANCH NEAR NASELLE, WA",049
3,USGS-12009525,"BEAR BRANCH AT RM 0.6 NEAR NASELLE, WA",049
4,USGS-12009600,"INDIAN CREEK NEAR CHINOOK, WA",049


### Peak Rise per Gauge
```daily``` is filtered by the same county expression plus the parameter, statistic, and incident window. The values come back without the location metadata, so we join them back to the gauge list, and the gauge list back to the declaration.

In [24]:
whereDaily = (whereCounties + " AND parameter_code='00065' AND statistic_id='00003'"
              " AND time >= DATE('" + floodBegin + "') AND time <= DATE('" + floodEnd + "')")
dailyJson = requests.get(waterBase + "daily/items",
                         params={"filter": whereDaily, "limit": 10000, "f": "json"}).json()

dfDaily = pd.DataFrame([f["properties"] for f in dailyJson["features"]])
dfDaily["value"] = pd.to_numeric(dfDaily["value"])
dfDaily = dfDaily.sort_values("time")

# rise cancels out each gauge's datum
dfRise = dfDaily.groupby("monitoring_location_id").agg(
    startStage=("value", "first"), peakStage=("value", "max")).reset_index()
dfRise["peakDate"] = dfDaily.loc[dfDaily.groupby("monitoring_location_id")["value"].idxmax()] \
                            .set_index("monitoring_location_id") \
                            .loc[dfRise["monitoring_location_id"], "time"].values
dfRise["riseFt"] = round(dfRise["peakStage"] - dfRise["startStage"], 2)

dfRise = dfRise.merge(dfGauges, on="monitoring_location_id") \
               .merge(dfFlood[["fipsCountyCode", "designatedArea"]], on="fipsCountyCode")

print(str(len(dfDaily)) + " daily gage-height readings from " + str(len(dfRise)) + " gauges")
dfRise[["designatedArea", "gauge", "peakDate", "startStage", "peakStage", "riseFt"]] \
    .sort_values("riseFt", ascending=False).head(10)

2541 daily gage-height readings from 173 gauges


,designatedArea,gauge,peakDate,startStage,peakStage,riseFt
59,King (County),"GREEN RIVER BELOW MEEKER ST AT KENT, WA",2025-12-14,21.41,43.64,22.23
95,King (County),"SNOQUALMIE RIVER AT DUVALL, WA",2025-12-11,22.46,43.97,21.51
123,Skagit (County),"SKAGIT RIVER NEAR MOUNT VERNON, WA",2025-12-12,14.21,35.06,20.85
121,Skagit (County),"SKAGIT RIVER NEAR CONCRETE, WA",2025-12-11,17.47,37.50,20.03
96,Snohomish (County),"SNOHOMISH RIVER NEAR MONROE, WA",2025-12-11,3.24,23.26,20.02
98,Snohomish (County),"SNOHOMISH RIVER AT SNOHOMISH, WA",2025-12-11,14.14,33.54,19.40
122,Skagit (County),"SKAGIT RIVER NEAR SEDRO WOOLLEY, WA",2025-12-11,25.56,42.22,16.66
120,Skagit (County),"SKAGIT RIVER ABOVE MILLER CREEK NEAR ROCKPORT, WA",2025-12-11,2.03,18.06,16.03
58,King (County),"GREEN RIVER ABOVE S 277TH ST AT KENT, WA",2025-12-18,39.37,54.06,14.69
141,Whatcom (County),"NOOKSACK RIVER AT FERNDALE, WA",2025-12-11,5.92,20.46,14.54



## Where to Go Next
Many of the examples above can be enhanced by charting or mapping the data. A future tutorial may illustrate this. One existing example is the ```get_data_from_api_display_on_map.ipynb``` notebook in the <a href='https://github.com/FEMA/openfema-samples/tree/master/analysis-examples'>OpenFEMA GitHub analysis-examples folder</a>.It joins OpenFEMA data with Census data and renders the result as a county-level choropleth map. 

The examples in this notebook utilize the ```pandas``` Python library. Other libraries exist that provide the same functionality. If you find SQL more useful, the [**DuckDB**](https://duckdb.org/) library is worth a look. It's an in-process database that facilitates data retrieval and joining using standard SQL. 

## Other Resources
- [OpenFEMA Homepage](https://www.fema.gov/open)
- [OpenFEMA API Documentation](https://www.fema.gov/about/openfema/api)
- [OpenFEMA Code Samples on GitHub](https://github.com/FEMA/openfema-samples/tree/master/code-samples)
- [U.S. Census Bureau API](https://www.census.gov/data/developers/data-sets.html)
- [Request a Census Bureau API Key](https://api.census.gov/data/key_signup.html)
- [Census TIGERweb REST Services](https://tigerweb.geo.census.gov/tigerwebmain/TIGERweb_restmapservice.html)
- [SAM.gov Entity Management API](https://open.gsa.gov/api/entity-api/)
- [USAspending.gov API Documentation](https://api.usaspending.gov/)
- [Federal Register API Documentation](https://www.federalregister.gov/developers/documentation/api/v1)
- [Weather.gov API Documentation](https://www.weather.gov/documentation/services-web-api)
- [Weather.gov Alerts API (alert records and parameters)](https://www.weather.gov/documentation/services-web-api#/default/alerts_query)
- [NWS Geolocation Documentation (PDF)](https://www.weather.gov/media/documentation/docs/NWS_Geolocation.pdf)
- [NOAA Storm Events Database (historical severe weather)](https://www.ncdc.noaa.gov/stormevents/)
- [NOAA Storm Events bulk CSV files](https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/)
- [USGS FDSN Earthquake Event API](https://earthquake.usgs.gov/fdsnws/event/1/)
- [USGS Water Data APIs](https://api.waterdata.usgs.gov/)
- [DuckDB](https://duckdb.org/)